# DSA 504 — Class 8
## Visualization I: matplotlib

**Date:** Monday, Sep 28
**Reading:** *Python for Data Analysis*, ch. 9

---

Starting today, we use `retail_sales_clean.csv` — the fully cleaned dataset from Class 7 (no missing values, no duplicates, exactly 5 real categories). This is the first class all semester working with trustworthy data from the start.

### Learning goals
By the end of this class, you will be able to:
- Explain the basic anatomy of a matplotlib figure (figure, axes)
- Create line plots, bar plots, histograms, and scatter plots
- Label axes, add titles and legends, and control figure size
- Plot multiple series on one chart, and create multi-panel figures with subplots
- Choose an appropriate chart type for a given question
- Save a figure to a file


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

sales = pd.read_csv("retail_sales_clean.csv")
sales["date"] = pd.to_datetime(sales["date"])
sales.head()


## 1. Why Visualize?

Everything so far — `.describe()`, `.groupby()`, `.sum()` — gives you numbers. Numbers are precise, but they're slow to absorb and easy to misread. A chart shows a pattern (a trend, an outlier, a comparison) **instantly**, in a way a table of numbers usually can't.

**matplotlib** is the foundational Python plotting library — nearly every other Python visualization tool (including seaborn, which we cover next class) is either built on top of it or works alongside it.


## 2. The Anatomy of a matplotlib Figure

Every matplotlib chart has two key objects:
- The **figure** — the overall canvas/window that holds everything
- The **axes** — the actual plot area inside the figure (confusingly, this means one chart, not the x/y axis lines)

The standard, recommended way to start any plot is `plt.subplots()`, which hands you both objects at once.


In [ ]:
fig, ax = plt.subplots()
print(type(fig))
print(type(ax))


You'll see this exact pattern — `fig, ax = plt.subplots()` — at the start of almost every plot we make this semester. Get comfortable typing it.


## 3. Line Plots — Showing a Trend Over Time

Line plots are the natural choice whenever your x-axis is something ordered, like time.


In [ ]:
# Total revenue per day, across all stores
daily_revenue = sales.groupby("date")["revenue"].sum()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(daily_revenue.index, daily_revenue.values)
ax.set_title("Total Daily Revenue, 2025")
ax.set_xlabel("Date")
ax.set_ylabel("Revenue ($)")
plt.show()


**Talking point:** notice how much easier it is to see the overall pattern (and the day-to-day noise) in this chart than it would be scrolling through 365 individual daily totals in a table.


In [ ]:
# Smoothing out the noise with a rolling average -- a common technique for noisy daily data
daily_revenue_smooth = daily_revenue.rolling(window=7).mean()   # 7-day rolling average

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(daily_revenue.index, daily_revenue.values, alpha=0.3, label="Daily")
ax.plot(daily_revenue_smooth.index, daily_revenue_smooth.values, label="7-day average", linewidth=2)
ax.set_title("Total Daily Revenue, with 7-Day Rolling Average")
ax.set_xlabel("Date")
ax.set_ylabel("Revenue ($)")
ax.legend()
plt.show()


## 4. Bar Plots — Comparing Categories

Bar plots are the right choice when you're comparing totals or averages *across distinct groups* — stores, categories, and so on — rather than showing a trend.


In [ ]:
revenue_by_store = sales.groupby("store")["revenue"].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(revenue_by_store.index, revenue_by_store.values)
ax.set_title("Total Revenue by Store")
ax.set_xlabel("Store")
ax.set_ylabel("Revenue ($)")
plt.show()


In [ ]:
# Horizontal bar plots (ax.barh) are often more readable when category labels are long
revenue_by_category = sales.groupby("category")["revenue"].sum().sort_values()

fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(revenue_by_category.index, revenue_by_category.values)
ax.set_title("Total Revenue by Category")
ax.set_xlabel("Revenue ($)")
plt.show()


**Chart selection note:** always sort bar charts by value (as done above) unless there's a specific reason not to (like a natural category order such as weekdays) — an unsorted bar chart makes visual comparison harder than it needs to be.


## 5. Histograms — Showing a Distribution

A histogram answers a different question than a bar chart: not "how do these groups compare," but "what does the *spread* of a single numeric column look like?"


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(sales["units_sold"], bins=30)
ax.set_title("Distribution of Units Sold per Transaction")
ax.set_xlabel("Units Sold")
ax.set_ylabel("Frequency (number of records)")
plt.show()


**The `bins` parameter matters more than it looks.** Too few bins hides real structure in the data; too many makes it look noisy. There's no single correct number — try a few values and see what best represents the actual shape of your data.


In [ ]:
# Comparing a few different bin counts side by side
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, n_bins in zip(axes, [10, 30, 100]):
    ax.hist(sales["units_sold"], bins=n_bins)
    ax.set_title(f"bins={n_bins}")
    ax.set_xlabel("Units Sold")

plt.tight_layout()
plt.show()


## 6. Scatter Plots — Showing a Relationship Between Two Variables

Scatter plots answer "does X relate to Y?" — one point per record, positioned by its two values.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(sales["units_sold"], sales["revenue"], alpha=0.3, s=10)
ax.set_title("Units Sold vs. Revenue")
ax.set_xlabel("Units Sold")
ax.set_ylabel("Revenue ($)")
plt.show()


**Why `alpha=0.3` here?** With thousands of points, a scatter plot often becomes a solid blob. Making each point partly transparent (`alpha`) lets overlapping points show up as darker regions — a simple way to reveal density without switching chart types.


## 7. Multiple Charts in One Figure

`plt.subplots(rows, cols)` creates a grid of charts sharing one figure — useful for comparing several views of the data at once.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left panel
revenue_by_store = sales.groupby("store")["revenue"].sum().sort_values(ascending=False)
axes[0].bar(revenue_by_store.index, revenue_by_store.values)
axes[0].set_title("Revenue by Store")
axes[0].set_ylabel("Revenue ($)")

# Right panel
axes[1].hist(sales["units_sold"], bins=30)
axes[1].set_title("Distribution of Units Sold")
axes[1].set_xlabel("Units Sold")

plt.tight_layout()
plt.show()


## 8. Saving a Figure to a File


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(revenue_by_store.index, revenue_by_store.values)
ax.set_title("Total Revenue by Store")
ax.set_ylabel("Revenue ($)")

fig.savefig("revenue_by_store.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to revenue_by_store.png")


`dpi=150` controls image resolution (higher = sharper, larger file), and `bbox_inches="tight"` trims excess whitespace around the figure — a small detail, but it noticeably improves how a saved chart looks when dropped into a report or slide.


## 9. Choosing the Right Chart Type

| Question you're asking | Chart type |
|---|---|
| How does this change over time? | Line plot |
| How do these groups compare? | Bar plot |
| What does the spread of one variable look like? | Histogram |
| Is there a relationship between two variables? | Scatter plot |

This table is worth returning to before making any chart — picking the wrong type for the question is one of the most common (and easily avoidable) visualization mistakes.


---
## Guided Practice

Work through these using `sales`, already loaded above from `retail_sales_clean.csv`.


### Exercise 1 — Line plot
Create a line plot of total daily `units_sold` (not revenue) across all stores, for the full year. Label the axes and add a title.


In [ ]:
# Exercise 1 — your code here



### Exercise 2 — Bar plot
Create a bar plot showing average `revenue` per category (not total — use `.mean()`), sorted from highest to lowest.


In [ ]:
# Exercise 2 — your code here



### Exercise 3 — Histogram
Create a histogram of the `revenue` column with 40 bins. Based on the shape, describe in a one-line code comment whether the distribution looks roughly symmetric or skewed in one direction.


In [ ]:
# Exercise 3 — your code here



### Exercise 4 — Scatter plot
Create a scatter plot of `units_sold` vs. `revenue`, but this time filtered to just one store of your choice. Use `alpha` to handle overlapping points.


In [ ]:
# Exercise 4 — your code here



### Exercise 5 (stretch) — Multi-panel comparison
Create a figure with a 2x2 grid of subplots (`plt.subplots(2, 2, ...)`) showing: total revenue by store (top-left), total revenue by category (top-right), a histogram of units_sold (bottom-left), and a scatter plot of units_sold vs. revenue (bottom-right). Give the whole figure a title using `fig.suptitle(...)`.


In [ ]:
# Exercise 5 — your code here



---
## Wrap-up

**Recap:** the figure/axes structure behind every matplotlib chart; line plots (trends), bar plots (comparisons), histograms (distributions), and scatter plots (relationships); labeling and titling charts properly; multi-panel figures with `plt.subplots()`; and saving a figure with `fig.savefig()`.

**Common mistakes to watch for:**
- Using a bar chart to show a trend over time, or a line chart to compare unrelated categories — match the chart type to the actual question
- Forgetting axis labels and titles — a chart without labels forces the viewer to guess what they're looking at
- An unsorted bar chart, making comparison harder than necessary
- Too many or too few histogram bins, obscuring the real shape of the distribution

**HW2 is in progress** (assigned Class 7, due before Class 9).

**Before next class (Sep 30):** Class 9 covers seaborn — a library built on top of matplotlib that makes many of today's charts easier to produce and adds a few new chart types better suited for exploratory analysis. **HW2 is due before Class 9.**
